In [1]:
# Cell 1a
import sys
sys.stderr = open("mcp_server.log", "w")

In [2]:
# Cell 1b
##import sys

##MCP_DIAGNOSTIC_LOG = open("mcp_server.log", "w", encoding="utf-8")
##sys.stderr = MCP_DIAGNOSTIC_LOG
##print("MCP server diagnostics will be written to mcp_server.log")

In [3]:
!pip install -q  fastmcp

In [4]:
# Cell 2a
%%writefile math_server.py
from fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def safe_calculator(expression: str) -> str:
    """Evaluate a basic math expression, such as '8273 * 1821'.
    Supports +, -, *, /, parentheses, and numbers only."""
    try:
        result = eval(expression, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing math_server.py


In [5]:
# Cell 2b
import sys
from pathlib import Path
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

transport = StdioTransport(
    command=sys.executable,
    args=[str(Path("math_server.py").resolve())]
)

client = Client(transport)

async with client:
    # Discover tools
    tools = await client.list_tools()
    print("Available tools:", [t.name for t in tools])

    # Call tool through MCP
    result = await client.call_tool(
        "safe_calculator",
        {"expression": "8273 * 1821"}
    )

    print("Result:", result.data)

Available tools: ['safe_calculator']
Result: 15065133
